In [2]:

import sys
import subprocess
import json
import textwrap
from pathlib import Path
from datetime import datetime
from xml.sax.saxutils import escape

try:
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "pandas", "reportlab"
    ])
    import pandas as pd
    from reportlab.lib import colors
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_JUSTIFY
    from reportlab.lib.units import inch
    from reportlab.platypus import (
        SimpleDocTemplate,
        Paragraph,
        Spacer,
        Table,
        TableStyle,
        PageBreak,
        Preformatted,
    )

# ============================================================
# 1) PROJECT ROOT + FIXED TARGETS
# ============================================================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_PATH = PROJECT_ROOT / "src" / "06-orchestration" / "pipeline_orchestration_prefect.ipynb"
ORCH_LOG_DIR = PROJECT_ROOT / "logs" / "orchestration"
VALIDATION_DIR = PROJECT_ROOT / "reports" / "validation"

OUTPUT_PATH = PROJECT_ROOT / "10 Pipeline Orchestration- DM4ML-Group51.pdf"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("NOTEBOOK_PATH:", NOTEBOOK_PATH)
print("ORCH_LOG_DIR:", ORCH_LOG_DIR)
print("OUTPUT_PATH:", OUTPUT_PATH)

# ============================================================
# 2) HELPERS
# ============================================================
def rel_path(path):
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT.resolve()))
    except Exception:
        return str(path)

def file_info(path):
    if not path or not Path(path).exists():
        return None
    p = Path(path)
    st = p.stat()
    return {
        "name": p.name,
        "relative_path": rel_path(p),
        "last_modified": datetime.fromtimestamp(st.st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "size_kb": round(st.st_size / 1024, 2),
        "suffix": p.suffix.lower(),
    }

def latest_files(folder, limit=3, patterns=None):
    folder = Path(folder)
    if not folder.exists():
        return []
    patterns = patterns or ["*.log", "*.txt", "*.jsonl", "*.out"]
    hits = []
    for pattern in patterns:
        hits.extend(folder.glob(pattern))
    hits = [p for p in hits if p.is_file()]
    hits.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return hits[:limit]

def read_text_file(path, max_chars=None):
    if not path or not Path(path).exists():
        return ""
    text = Path(path).read_text(encoding="utf-8", errors="ignore")
    return text[:max_chars] if max_chars else text

def read_text_preview(path, max_lines=120, max_chars=15000):
    if not path or not Path(path).exists():
        return "File not found."
    text = read_text_file(path)
    return "\n".join(text.splitlines()[:max_lines])[:max_chars]

def read_notebook_code(path, max_chars=50000):
    if not path or not Path(path).exists():
        return "Notebook file not found."
    try:
        nb = json.loads(Path(path).read_text(encoding="utf-8", errors="ignore"))
        parts = []
        for idx, cell in enumerate(nb.get("cells", []), start=1):
            ctype = cell.get("cell_type", "")
            src = cell.get("source", [])
            src = "".join(src) if isinstance(src, list) else str(src)
            src = src.rstrip()

            if ctype == "markdown" and src.strip():
                parts.append(f"# ---- Markdown Cell {idx} ----\n{src}")
            elif ctype == "code" and src.strip():
                parts.append(f"# ---- Code Cell {idx} ----\n{src}")

        combined = "\n\n".join(parts).strip()
        return combined[:max_chars] if combined else "No notebook content found."
    except Exception as e:
        return f"Could not parse notebook: {e}"

def wrap_block_text(text, width=95):
    wrapped = []
    for line in str(text).splitlines():
        if not line.strip():
            wrapped.append("")
            continue
        pieces = textwrap.wrap(
            line,
            width=width,
            break_long_words=True,
            break_on_hyphens=True,
            replace_whitespace=False,
            drop_whitespace=False,
        )
        wrapped.extend(pieces if pieces else [""])
    return "\n".join(wrapped)

def wrap_path_for_pdf(value, max_chunk=32):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    separators = {"\\", "/", "_", "-", "="}
    parts = []
    token = ""

    for ch in text:
        token += ch
        if ch in separators:
            parts.append(token)
            token = ""
    if token:
        parts.append(token)

    lines = []
    current = ""

    for part in parts:
        if len(current) + len(part) <= max_chunk:
            current += part
        else:
            if current:
                lines.append(current)
            if len(part) <= max_chunk:
                current = part
            else:
                subparts = textwrap.wrap(
                    part,
                    width=max_chunk,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if subparts:
                    lines.extend(subparts[:-1])
                    current = subparts[-1]
                else:
                    current = part

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def wrap_general_text_for_pdf(value, max_len=38):
    if value is None:
        return ""
    text = str(value).strip()
    if not text:
        return ""

    words = text.split()
    lines = []
    current = ""

    for word in words:
        candidate = f"{current} {word}".strip()
        if len(candidate) <= max_len:
            current = candidate
        else:
            if current:
                lines.append(current)
            if len(word) > max_len:
                chunks = textwrap.wrap(
                    word,
                    width=max_len,
                    break_long_words=True,
                    break_on_hyphens=True,
                )
                if chunks:
                    lines.extend(chunks[:-1])
                    current = chunks[-1]
                else:
                    current = word
            else:
                current = word

    if current:
        lines.append(current)

    return "<br/>".join(escape(x) for x in lines)

def make_display_value(v):
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    return str(v)

# ============================================================
# 3) LOAD NOTEBOOK + LATEST ORCHESTRATION LOGS
# ============================================================
notebook_text = read_notebook_code(NOTEBOOK_PATH, max_chars=120000)
latest_orch_logs = latest_files(ORCH_LOG_DIR, limit=5, patterns=["*.log", "*.txt", "*.jsonl", "*.out"])

log_sections = []
for p in latest_orch_logs:
    log_sections.append({
        "path": p,
        "preview": read_text_preview(p, max_lines=160, max_chars=18000),
    })

latest_validation_txt = None
candidate_validation = PROJECT_ROOT / "src" / "02-validation" / "run_validation.txt"
if candidate_validation.exists():
    latest_validation_txt = candidate_validation

# ============================================================
# 4) SUMMARY TABLE DATA
# ============================================================
team_data = [
    ["Team Member Name", "Team Member ID"],
    ["BANSHIDHAR RATH", "2025AE05346"],
    ["JITENDRA KUMAR TIWARI", "2025AE05518"],
    ["KATBA ANKIT CHIMANBHAI", "2025AE05229"],
    ["NAVEEN SURATHU", "2025AE05492"],
]

artifact_rows = [["Artifact", "Relative Path", "Last Modified", "Size"]]
for p in [NOTEBOOK_PATH, latest_validation_txt, *latest_orch_logs]:
    if p and Path(p).exists():
        info = file_info(p)
        artifact_rows.append([
            info["name"],
            info["relative_path"],
            info["last_modified"],
            f"{info['size_kb']} KB",
        ])
if len(artifact_rows) == 1:
    artifact_rows.append(["No artifacts found", "-", "-", "-"])

orch_summary_df = pd.DataFrame([{
    "orchestration_tool": "Prefect",
    "notebook_found": NOTEBOOK_PATH.exists(),
    "notebook_path": rel_path(NOTEBOOK_PATH),
    "log_folder_found": ORCH_LOG_DIR.exists(),
    "latest_logs_found": len(latest_orch_logs),
}])

logs_summary_rows = [["Log File", "Relative Path", "Last Modified", "Size"]]
for p in latest_orch_logs:
    info = file_info(p)
    logs_summary_rows.append([
        info["name"],
        info["relative_path"],
        info["last_modified"],
        f"{info['size_kb']} KB",
    ])
if len(logs_summary_rows) == 1:
    logs_summary_rows.append(["No orchestration logs found", "-", "-", "-"])

# ============================================================
# 5) PDF STYLES
# ============================================================
styles = getSampleStyleSheet()

title_style = ParagraphStyle(
    name="CustomTitle",
    parent=styles["Title"],
    alignment=TA_CENTER,
    fontSize=16,
    leading=20,
    spaceAfter=14,
)

meta_style = ParagraphStyle(
    name="MetaStyle",
    parent=styles["Normal"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=13,
    spaceAfter=5,
)

heading_style = ParagraphStyle(
    name="HeadingStyle",
    parent=styles["Heading2"],
    alignment=TA_LEFT,
    fontSize=12,
    leading=15,
    spaceAfter=8,
)

sub_heading_style = ParagraphStyle(
    name="SubHeadingStyle",
    parent=styles["Heading3"],
    alignment=TA_LEFT,
    fontSize=10.2,
    leading=12.5,
    spaceAfter=6,
)

body_style = ParagraphStyle(
    name="BodyStyle",
    parent=styles["BodyText"],
    alignment=TA_JUSTIFY,
    fontSize=10.0,
    leading=14,
    spaceAfter=8,
)

bullet_style = ParagraphStyle(
    name="BulletStyle",
    parent=styles["BodyText"],
    alignment=TA_LEFT,
    fontSize=10.0,
    leading=14,
    leftIndent=14,
    firstLineIndent=-8,
    spaceAfter=4,
)

code_style = ParagraphStyle(
    name="CodeStyle",
    parent=styles["Code"],
    fontName="Courier",
    fontSize=7.0,
    leading=8.4,
)

table_header_style = ParagraphStyle(
    name="TableHeaderStyle",
    parent=styles["BodyText"],
    fontName="Helvetica-Bold",
    fontSize=8.0,
    leading=9.2,
    alignment=TA_LEFT,
)

table_cell_style = ParagraphStyle(
    name="TableCellStyle",
    parent=styles["BodyText"],
    fontName="Helvetica",
    fontSize=7.0,
    leading=8.4,
    alignment=TA_LEFT,
)

def to_para(value, style, kind="general"):
    if kind == "path":
        return Paragraph(wrap_path_for_pdf(value), style)
    return Paragraph(wrap_general_text_for_pdf(value), style)

def make_wrapped_table(data, col_widths=None, path_cols=None):
    path_cols = path_cols or []
    converted = []

    for r, row in enumerate(data):
        row_cells = []
        for c, cell in enumerate(row):
            st = table_header_style if r == 0 else table_cell_style
            if r == 0:
                row_cells.append(Paragraph(escape(str(cell)), st))
            else:
                row_cells.append(to_para(cell, st, "path" if c in path_cols else "general"))
        converted.append(row_cells)

    table = Table(converted, colWidths=col_widths, repeatRows=1)
    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#D9EAD3")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.grey),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 6),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
    ]))
    return table

def df_to_wrapped_table(df, max_rows=20, col_widths=None, path_cols=None):
    if df is None or df.empty:
        data = [["No data available"]]
    else:
        preview = df.head(max_rows).copy()
        for col in preview.columns:
            preview[col] = preview[col].map(make_display_value)
        data = [list(preview.columns)] + preview.astype(str).values.tolist()
    return make_wrapped_table(data, col_widths=col_widths, path_cols=path_cols)

# ============================================================
# 6) BUILD PDF STORY
# ============================================================
story = []

story.append(Paragraph("10 Pipeline Orchestration", title_style))
story.append(Paragraph("<b>Course Name:</b> Data Management for Machine Learning", meta_style))
story.append(Paragraph("<b>Assignment Title:</b> End-to-End Data Management Pipeline for a Recommendation System", meta_style))
story.append(Paragraph("<b>Assignment:</b> Group 51 - Data management for Machine Learning Group 51", meta_style))
story.append(Spacer(1, 10))

story.append(Paragraph("<b>Team Members</b>", heading_style))
story.append(make_wrapped_table(team_data, col_widths=[4.0 * inch, 2.0 * inch]))
story.append(Spacer(1, 12))

story.append(Paragraph("1. Objective", heading_style))
story.append(Paragraph(
    "This report documents pipeline orchestration for the recommendation pipeline by embedding the Prefect notebook content from src\\06-orchestration\\pipeline_orchestration_prefect.ipynb and printing the latest orchestration logs from logs\\orchestration.",
    body_style
))

story.append(Paragraph("2. Objective Coverage", heading_style))
for note in [
    "Automate the end-to-end pipeline using Prefect.",
    "Capture orchestration code for ingestion, validation, preparation, transformation, feature store, and model training.",
    "Print latest orchestration execution logs.",
    "Render long text as wrapped Paragraph cells.",
    "Wrap long log lines before passing them to Preformatted.",
    "Break long file paths only at safe separators such as \\, /, _, -, and =.",
]:
    story.append(Paragraph(f"• {escape(note)}", bullet_style))
story.append(Spacer(1, 12))

story.append(Paragraph("3. Supporting Artifacts", heading_style))
story.append(make_wrapped_table(
    artifact_rows,
    col_widths=[1.55 * inch, 3.35 * inch, 1.00 * inch, 0.60 * inch],
    path_cols=[1]
))
story.append(Spacer(1, 12))

story.append(Paragraph("4. Orchestration Summary", heading_style))
story.append(df_to_wrapped_table(
    orch_summary_df,
    max_rows=10,
    path_cols=[2]
))
story.append(Spacer(1, 12))

story.append(Paragraph("5. Latest Orchestration Logs Summary", heading_style))
story.append(make_wrapped_table(
    logs_summary_rows,
    col_widths=[1.55 * inch, 3.25 * inch, 1.00 * inch, 0.70 * inch],
    path_cols=[1]
))
story.append(PageBreak())

story.append(Paragraph("6. Orchestration DAG / Notebook Content", heading_style))
story.append(Paragraph(
    f"<b>Path:</b> {escape(rel_path(NOTEBOOK_PATH))}",
    meta_style
))
story.append(Preformatted(wrap_block_text(notebook_text, width=95), code_style))
story.append(PageBreak())

story.append(Paragraph("7. Latest Orchestration Logs", heading_style))
if log_sections:
    for item in log_sections:
        story.append(Paragraph(f"Log File: {escape(item['path'].name)}", sub_heading_style))
        story.append(Paragraph(f"<b>Path:</b> {escape(rel_path(item['path']))}", meta_style))
        story.append(Preformatted(wrap_block_text(item["preview"], width=95), code_style))
        story.append(Spacer(1, 10))
else:
    story.append(Paragraph("No files were found in logs\\orchestration.", body_style))

if latest_validation_txt and latest_validation_txt.exists():
    story.append(Paragraph("8. Additional Pipeline Evidence", heading_style))
    story.append(Paragraph(
        "The validation workflow output is included as supporting pipeline evidence when available.",
        body_style
    ))
    story.append(Preformatted(
        wrap_block_text(read_text_preview(latest_validation_txt, max_lines=80, max_chars=10000), width=95),
        code_style
    ))
    story.append(Spacer(1, 10))

story.append(Paragraph("9. Conclusion", heading_style))
story.append(Paragraph(
    "This PDF consolidates the pipeline orchestration deliverables by embedding the specified Prefect notebook content and printing the latest orchestration execution logs in a submission-ready format.",
    body_style
))

# ============================================================
# 7) BUILD PDF
# ============================================================
doc = SimpleDocTemplate(
    str(OUTPUT_PATH),
    pagesize=A4,
    rightMargin=0.50 * inch,
    leftMargin=0.50 * inch,
    topMargin=0.55 * inch,
    bottomMargin=0.55 * inch,
)
doc.build(story)

print(f"PDF created successfully: {OUTPUT_PATH}")
print(f"Notebook exists: {NOTEBOOK_PATH.exists()}")
print(f"Latest orchestration logs found: {len(latest_orch_logs)}")


PROJECT_ROOT: C:\Users\barath\recomart-pipeline
NOTEBOOK_PATH: C:\Users\barath\recomart-pipeline\src\06-orchestration\pipeline_orchestration_prefect.ipynb
ORCH_LOG_DIR: C:\Users\barath\recomart-pipeline\logs\orchestration
OUTPUT_PATH: C:\Users\barath\recomart-pipeline\10 Pipeline Orchestration- DM4ML-Group51.pdf
PDF created successfully: C:\Users\barath\recomart-pipeline\10 Pipeline Orchestration- DM4ML-Group51.pdf
Notebook exists: True
Latest orchestration logs found: 5
